# EDA — Historical Streamflow Series: Itajaí-Açu River at Blumenau

**ANA Station:** 83500000 — Blumenau  
**Variables:** streamflow (m³/s) and stage (m)  
**Objective:** understand the distribution, seasonality, extreme events and gaps
before training any model.

---

### Why perform EDA before modeling?

Machine learning models are loss-function optimizers — they
do not "know" that a value of -999 is a missing-data code, or that a
streamflow of 50,000 m³/s is physically impossible for this basin. If you
feed dirty data, the model learns dirty patterns.
This EDA has three concrete objectives:
1. Validate that we downloaded what we expected (sanity check)
2. Identify data quality issues (outliers, gaps, sensor changes)
3. Inform preprocessing decisions

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from pathlib import Path

# Consistent style across all plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGDIR = Path('../../reports/figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

STATION = '83500000'
RAW_DIR = Path('../../data/raw/streamflow')

# Critical flood events to annotate on plots
FLOOD_EVENTS = {
    '1983-11-09': ('Nov/1983', '17.17 m'),
    '1984-07-07': ('Jul/1984', '16.42 m'),
    '2008-11-23': ('Nov/2008', '11.78 m'),
    '2011-09-08': ('Sep/2011', '10.14 m'),
    '2023-09-05': ('Sep/2023', '10.55 m'),
}

print('Environment configured.')

## 1. Data Download

If the files already exist in `data/raw/streamflow/`, this cell
is a no-op. Otherwise, it downloads the full series.

In [ ]:
from src.data.ana_downloader import download_series, save_raw

vazao_path = RAW_DIR / f'{STATION}_vazao_raw.parquet'
cota_path  = RAW_DIR / f'{STATION}_cota_raw.parquet'

if not vazao_path.exists():
    print('Downloading streamflow series...')
    df_vazao = download_series(STATION, 'vazao', start_year=1940)
    save_raw(df_vazao, STATION, 'vazao')
else:
    print(f'Loading streamflow from {vazao_path}')
    df_vazao = pd.read_parquet(vazao_path)

if not cota_path.exists():
    print('Downloading stage series...')
    df_cota = download_series(STATION, 'cota', start_year=1940)
    save_raw(df_cota, STATION, 'cota')
else:
    print(f'Loading stage from {cota_path}')
    df_cota = pd.read_parquet(cota_path)

# Alias for the main analysis variable
Q = df_vazao['value'].rename('Q_m3s')
H = df_cota['value'].rename('H_m')

print(f'\nStreamflow: {Q.index.min().date()} → {Q.index.max().date()} ({len(Q)} days)')
print(f'Stage:      {H.index.min().date()} → {H.index.max().date()} ({len(H)} days)')

## 2. Sanity Check — Basic Statistics

Before any plot: numbers that tell us whether the download makes sense.

In [ ]:
print('=== STREAMFLOW (m³/s) ===')
print(Q.describe().round(1))
print(f'\nNaN: {Q.isna().sum()} ({Q.isna().mean()*100:.1f}%)')
print(f'Zeros: {(Q == 0).sum()}')
print(f'Negatives: {(Q < 0).sum()}  ← should be 0')
print(f'> 15000 m³/s: {(Q > 15000).sum()}  ← physically impossible')

print('\n=== STAGE (m) ===')
print(H.describe().round(2))
print(f'\nNaN: {H.isna().sum()} ({H.isna().mean()*100:.1f}%)')
print(f'Expected historical maximum: ~17.17 m (Nov/1983)')
print(f'Maximum found: {H.max():.2f} m')

## 3. Full Historical Series

Line plot with annotated critical events. Look for:
- Abrupt changes in mean level (may indicate sensor replacement)
- Continuous blocks of NaN (periods without measurements)
- Peaks consistent with known historical events

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)

# Streamflow
ax = axes[0]
ax.plot(Q.index, Q.values, lw=0.5, color='steelblue', alpha=0.8)
ax.set_ylabel('Streamflow (m³/s)', fontsize=11)
ax.set_title('Itajaí-Açu River — Blumenau Station (83500000)', fontsize=13, fontweight='bold')

# Annotate events
for date_str, (label, cota_str) in FLOOD_EVENTS.items():
    try:
        date = pd.Timestamp(date_str)
        val = Q.get(date, np.nan)
        if not np.isnan(val):
            ax.annotate(
                f'{label}\nH={cota_str}',
                xy=(date, val), xytext=(0, 30),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='crimson'),
                fontsize=7, color='crimson', ha='center'
            )
    except Exception:
        pass

# Stage
ax2 = axes[1]
ax2.plot(H.index, H.values, lw=0.5, color='darkorange', alpha=0.8)
ax2.axhline(9.0, color='gold', lw=1, ls='--', label='Alert (9 m)')
ax2.axhline(11.5, color='red', lw=1, ls='--', label='Emergency (11.5 m)')
ax2.set_ylabel('Stage (m)', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGDIR / '01_serie_historica_completa.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Completeness Analysis — Gaps

Annual heatmap by month: missing data in red, present in green.

**Why does this matter for the model?**  
The LSTM needs continuous sequences to propagate the hidden state.
Decades with many gaps force sequence cuts, reducing the
effective training set. Knowing *where* the gaps are helps us
choose the train/val/test splits.

In [ ]:
# Pivot: rows = year, columns = month
completeness = (
    Q.resample('MS').apply(lambda s: s.notna().mean())
    .to_frame('completeness')
    .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
    .pivot(index='year', columns='month', values='completeness')
)
completeness.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                        'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, max(8, len(completeness) // 4)))
sns.heatmap(
    completeness, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Fraction of days with valid data'}
)
ax.set_title('Streamflow Series Completeness by Year/Month', fontsize=13)
ax.set_ylabel('Year')
ax.set_xlabel('')
fig.tight_layout()
fig.savefig(FIGDIR / '02_completude_anual.png', dpi=150, bbox_inches='tight')
plt.show()

# Numerical summary
annual_completeness = completeness.mean(axis=1)
bad_years = annual_completeness[annual_completeness < 0.8]
print(f'Years with < 80% completeness: {len(bad_years)}')
if not bad_years.empty:
    print(bad_years.round(2).to_string())

## 5. Distribution and Skewness

The streamflow distribution is fundamental for choosing the transformation.

**What we expect to see:** log-normal distribution — strongly right-skewed
in linear space, reasonably Gaussian in log space.
If that is not the case, we need to revise the normalization strategy.

In [ ]:
from scipy import stats

Q_clean = Q.dropna()
Q_log = np.log1p(Q_clean)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Linear histogram
axes[0].hist(Q_clean, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('Streamflow (m³/s)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Linear Space')
axes[0].set_yscale('log')
sk = stats.skew(Q_clean)
axes[0].text(0.97, 0.95, f'Skewness = {sk:.2f}', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.7))

# 2. Log histogram
axes[1].hist(Q_log, bins=60, color='seagreen', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('log(1 + Streamflow)')
axes[1].set_title('Log Space')
sk_log = stats.skew(Q_log)
axes[1].text(0.97, 0.95, f'Skewness = {sk_log:.2f}', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.7))

# 3. Log Q-Q plot
stats.probplot(Q_log, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (log space vs. Normal)')
axes[2].get_lines()[1].set_color('crimson')

fig.suptitle('Streamflow Distribution — Blumenau', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(FIGDIR / '03_distribuicao_vazao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Linear skewness: {sk:.2f} (ideal for log-transform: > 1)')
print(f'Log skewness:    {sk_log:.2f} (ideal after transform: close to 0)')

## 6. Seasonality

Monthly boxplot — reveals when floods are most likely.

**What we expect:**  
Blumenau has a subtropical regime: orographic rainfall concentrated in
October–March. We expect much taller boxplots with longer upper tails
during those months.

**Why does this matter for the model?**  
The MEF-LSTM does not receive month as an explicit feature, but it does
receive static basin attributes. Understanding seasonality tells us whether
we need to include some temporal encoding (sine/cosine of day-of-year) as
a hindcast feature.

In [ ]:
df_monthly = Q_clean.to_frame()
df_monthly['month'] = df_monthly.index.month
df_monthly['month_name'] = df_monthly.index.strftime('%b')

month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
df_monthly['month_name'] = pd.Categorical(df_monthly['month_name'], categories=month_order)

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=df_monthly, x='month_name', y='Q_m3s',
    ax=ax, flierprops=dict(marker='.', ms=2, alpha=0.3),
    color='steelblue', linewidth=0.8
)
ax.set_yscale('log')
ax.set_xlabel('')
ax.set_ylabel('Streamflow (m³/s) — log scale')
ax.set_title('Streamflow Seasonality — Blumenau (log scale)', fontsize=13)

# Highlight high-flow months
for m in [0, 1, 2, 9, 10, 11]:  # Jan, Feb, Mar, Oct, Nov, Dec
    ax.axvspan(m - 0.5, m + 0.5, alpha=0.06, color='salmon')

fig.tight_layout()
fig.savefig(FIGDIR / '04_sazonalidade_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

# Monthly medians
print('Median by month (m³/s):')
print(df_monthly.groupby('month')['Q_m3s'].median().round(1).to_string())

## 7. Autocorrelation

How long does the river's "memory" last?

**Why does this define `hindcast_length` in the YAML?**  
The LSTM uses a window of `hindcast_length` days as historical context.
If autocorrelation drops to zero after 7 days, there is no benefit in using
30 days of hindcast — the model would waste parameters trying to
learn from noise. The current window in the config is 168 h (7 daily
or hourly days) — let's verify whether that is adequate.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

Q_log_clean = Q_log.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(Q_log_clean, lags=30, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — Log Streamflow (lags in days)')
axes[0].set_xlabel('Lag (days)')

plot_pacf(Q_log_clean, lags=30, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('PACF — Log Streamflow (lags in days)')
axes[1].set_xlabel('Lag (days)')

fig.tight_layout()
fig.savefig(FIGDIR / '05_autocorrelacao.png', dpi=150, bbox_inches='tight')
plt.show()

# Lag at which ACF drops below 0.5
from statsmodels.tsa.stattools import acf
acf_values = acf(Q_log_clean, nlags=30, fft=True)
lag_half = next((i for i, v in enumerate(acf_values) if v < 0.5), None)
print(f'ACF drops below 0.5 at lag: {lag_half} days')
print('→ Suggests a hindcast of at least that many days.')

## 8. Extreme Event Analysis

Zoom into the five largest historical events. We observe:
- Rise rate (m³/s per day) — how much warning time does the model need to provide?
- Hydrograph shape — single peak vs. multiple peaks?
- Duration of flooding above the alert stage threshold

In [ ]:
WINDOW_DAYS = 30  # window around the peak

fig, axes = plt.subplots(
    len(FLOOD_EVENTS), 1,
    figsize=(14, 4 * len(FLOOD_EVENTS)),
    sharex=False
)

for ax, (date_str, (label, cota_str)) in zip(axes, FLOOD_EVENTS.items()):
    peak = pd.Timestamp(date_str)
    window = Q.loc[peak - pd.Timedelta(days=WINDOW_DAYS):peak + pd.Timedelta(days=WINDOW_DAYS)]

    if window.empty:
        ax.text(0.5, 0.5, 'Data unavailable', ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
        ax.set_title(f'{label} — no data')
        continue

    ax.fill_between(window.index, window.values, alpha=0.3, color='steelblue')
    ax.plot(window.index, window.values, color='steelblue', lw=1.5)
    ax.axvline(peak, color='crimson', lw=1.5, ls='--', label='Declared peak')

    # Maximum rise rate
    diff = window.diff()
    max_rise = diff.max()
    max_rise_date = diff.idxmax()

    ax.set_title(
        f'{label} | peak stage: {cota_str} | max Q: {window.max():.0f} m³/s '
        f'| max rise: +{max_rise:.0f} m³/s/day',
        fontsize=9
    )
    ax.set_ylabel('Q (m³/s)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.legend(fontsize=8)

    # Shade the period above alert level (proxy: Q > 1000 m³/s)
    ax.axhspan(1000, window.max() * 1.05, alpha=0.04, color='red')

fig.suptitle('Hydrographs of Extreme Events — Itajaí-Açu River / Blumenau',
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGDIR / '06_eventos_extremos.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Stage × Streamflow Curve (Rating Curve)

The stage-discharge relationship is the station's "calibration": it converts
water level (easy to measure) into streamflow (computed). Problems with this
curve appear as a scattered point cloud or regime shifts over time
(sediment removal, channel engineering works, etc.).

In [ ]:
# Align series by index
df_hq = pd.concat([H, Q], axis=1).dropna()

if len(df_hq) < 10:
    print('Insufficient data for stage-discharge curve. Check that both series were downloaded.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))

    # Color by decade to detect regime shifts
    df_hq['decade'] = (df_hq.index.year // 10) * 10
    cmap = plt.cm.get_cmap('plasma', df_hq['decade'].nunique())

    for i, (decade, grp) in enumerate(df_hq.groupby('decade')):
        ax.scatter(grp['H_m'], grp['Q_m3s'], s=2, alpha=0.4,
                   color=cmap(i), label=str(decade))

    ax.set_xlabel('Stage (m)')
    ax.set_ylabel('Streamflow (m³/s)')
    ax.set_yscale('log')
    ax.set_title('Stage × Discharge Curve by Decade', fontsize=12)
    ax.legend(title='Decade', fontsize=8, markerscale=4)
    fig.tight_layout()
    fig.savefig(FIGDIR / '07_curva_cota_descarga.png', dpi=150, bbox_inches='tight')
    plt.show()

    r = df_hq[['H_m', 'Q_m3s']].corr().loc['H_m', 'Q_m3s']
    print(f'H×Q correlation: {r:.3f}')

## 10. Annual Maximum Series (Frequency Analysis)

How frequent is a flood of the magnitude of 1983?  
We fit a GEV (Generalized Extreme Value) distribution — the most widely
used in flood frequency analysis. The estimated return period helps
contextualize model performance: missing a 100-year flood is far more
costly than missing a 2-year event.

In [ ]:
from scipy.stats import genextreme

# Annual maximum
Q_annual_max = Q_clean.resample('YE').max().dropna()
Q_annual_max = Q_annual_max[Q_annual_max > 0]

# GEV fit
shape, loc, scale = genextreme.fit(Q_annual_max, loc=Q_annual_max.mean())

# Return periods
T = np.array([2, 5, 10, 25, 50, 100, 200, 500])
p_exceed = 1 / T
Q_T = genextreme.ppf(1 - p_exceed, shape, loc, scale)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual maximum series
axes[0].bar(Q_annual_max.index.year, Q_annual_max.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Annual maximum Q (m³/s)')
axes[0].set_title('Annual Maximum Streamflow')
axes[0].axhline(Q_annual_max.mean(), color='k', ls='--', lw=1, label=f'Mean = {Q_annual_max.mean():.0f}')
axes[0].legend()

# GEV frequency curve
T_plot = np.logspace(0, 3, 200)
Q_plot = genextreme.ppf(1 - 1/T_plot, shape, loc, scale)
axes[1].semilogx(T_plot, Q_plot, color='steelblue', lw=2, label='Fitted GEV')

# Empirical plotting positions (Gringorten)
n = len(Q_annual_max)
ranks = Q_annual_max.rank()
T_emp = (n + 0.12) / (ranks - 0.44)
axes[1].scatter(T_emp, Q_annual_max.values, color='k', s=20, zorder=5, label='Observed')

for t_val, q_val in zip(T, Q_T):
    axes[1].annotate(f'T={t_val}yr\n{q_val:.0f}', xy=(t_val, q_val),
                     fontsize=7, ha='left', color='crimson')

axes[1].set_xlabel('Return Period (years)')
axes[1].set_ylabel('Streamflow (m³/s)')
axes[1].set_title('Frequency Analysis — GEV')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGDIR / '08_frequencia_cheias.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nGEV return period estimates:')
for t_val, q_val in zip(T, Q_T):
    print(f'  T={t_val:4d} years → Q = {q_val:7.0f} m³/s')

## 11. Preprocessing Decisions — Summary

Based on the EDA above, we document here the decisions for `preprocessor.py`:

In [ ]:
decisions = {
    'log_transform': {
        'decision': True,
        'rationale': 'Linear skewness > 1; log skewness close to 0 — confirms log-normal distribution.',
    },
    'max_gap_interpolate': {
        'decision': 3,
        'rationale': 'Short gaps (1–3 days) are likely sensor failures, not hydrological events. '
                     'Linear interpolation is conservative for that horizon.',
    },
    'train_split': {
        'decision': 'Train: 1940–2000 | Val: 2001–2010 | Test: 2011–2024',
        'rationale': 'Test set includes the 2011 and 2023 events. '
                     'Val set covers a reasonably complete data period before the most recent events.',
    },
    'normalization': {
        'decision': 'Z-score computed ONLY on the training period',
        'rationale': 'Data leakage: using statistics from the full dataset leaks val/test information.',
    },
    'consistency_priority': {
        'decision': 'Level 2 (consolidated) preferred over Level 1 (raw)',
        'rationale': 'Consolidated data have undergone manual review. '
                     'For recent dates without consolidation, accept Level 1.',
    },
}

for key, val in decisions.items():
    print(f'\n── {key} ──')
    print(f'  Decision:  {val["decision"]}')
    print(f'  Rationale: {val["rationale"]}')

## 12. Next Steps

With the EDA complete, the natural workflow is:

1. **`notebooks/02_preprocessing/`** — run `preprocessor.py` and visually
   validate the processed series against the raw series
2. **Precipitation data** — repeat the EDA for the rainfall stations
   across the basin (required as model features)
3. **Static basin attributes** — extract drainage area, slope,
   land cover via MERIT Hydro / MODIS
4. **Format for OpenHydroNet** — convert to NetCDF with the structure
   expected by the framework (one folder per basin, `timeseries.nc`
   and `attributes.csv` files)